# Extração de Padrões - Nação Nutrida

Este notebook apresenta a etapa de mineração de dados do projeto Nação Nutrida, utilizando o algoritmo Apriori para descoberta de padrões de associação entre alimentos doados em campanhas.

In [ ]:
# IMPORTAÇÃO DAS BIBLIOTECAS
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [ ]:
# CARREGAMENTO DOS DADOS
doacoes = pd.read_csv('../base_dados/alimento_doacao.csv')
alimentos = pd.read_csv('../base_dados/alimento.csv')

print(doacoes.head())
print(alimentos.head())

   cd_alimento_doacao  cd_usuario  cd_alimento  cd_campanha  qt_alimento_doado
0                   1           7           11            1                 25
1                   2           5           13            1                 30
2                   3           2            5            2                 50
3                   4          23           29            2                 10
4                   5          24            1            2                 40
   cd_alimento  nm_alimento sg_medida_alimento nm_tipo_alimento  \
0            1        Leite                  L      Mais doados   
1            2  Leite em pó                 KG      Mais doados   
2            3     Macarrão                 KG      Mais doados   
3            4         Café                 KG      Mais doados   
4            5       Açúcar                 KG      Mais doados   

   cd_tipo_alimento  
0                 1  
1                 1  
2                 1  
3                 1  
4            

In [ ]:
# LIMPEZA DOS DADOS
doacoes = doacoes.dropna()
alimentos = alimentos.dropna()

doacoes = doacoes.drop_duplicates()
alimentos = alimentos.drop_duplicates()

In [ ]:
# JUNÇÃO DOS DADOS
df = doacoes.merge(alimentos, on='cd_alimento')

print(df.head())

   cd_alimento_doacao  cd_usuario  cd_alimento  cd_campanha  \
0                   1           7           11            1   
1                   2           5           13            1   
2                   3           2            5            2   
3                   4          23           29            2   
4                   5          24            1            2   

   qt_alimento_doado nm_alimento sg_medida_alimento    nm_tipo_alimento  \
0                 25        Fubá                 KG         Mais doados   
1                 30      Feijão                 KG         Mais doados   
2                 50      Açúcar                 KG         Mais doados   
3                 10     Iogurte                  L  Derivados do leite   
4                 40       Leite                  L         Mais doados   

   cd_tipo_alimento  
0                 1  
1                 1  
2                 1  
3                 5  
4                 1  


In [ ]:
# CRIAÇÃO DAS TRANSAÇÕES
transacoes = df.groupby('cd_campanha')['nm_alimento'].apply(list).tolist()

print(transacoes[:5])

[['Fubá', 'Feijão'], ['Açúcar', 'Iogurte', 'Leite'], ['Cebola', 'Batata'], ['Maçã', 'Banana', 'Laranja'], ['Carne de Frango', 'Linguiça', 'Carne Bovina']]


In [ ]:
# MATRIZ BINÁRIA
te = TransactionEncoder()

te_array = te.fit(transacoes).transform(transacoes)

df_final = pd.DataFrame(te_array,
                        columns=te.columns_).astype(bool)

print(df_final.head())

   Abóbora  Alface  Arroz  Açúcar  Banana  Batata   Café  Carne Bovina  \
0    False   False  False   False   False   False  False         False   
1    False   False  False    True   False   False  False         False   
2    False   False  False   False   False    True  False         False   
3    False   False  False   False    True   False  False         False   
4    False   False  False   False   False   False  False          True   

   Carne de Frango  Cebola  ...  Linguiça  Macarrão  Mamão  Margarina   Maçã  \
0            False   False  ...     False     False  False      False  False   
1            False   False  ...     False     False  False      False  False   
2            False    True  ...     False     False  False      False  False   
3            False   False  ...     False     False  False      False   True   
4             True   False  ...      True     False  False      False  False   

    Ovos  Queijo    Sal  Tomate   Óleo  
0  False   False  False   False  

In [ ]:
# APLICAÇÃO DO APRIORI
frequentes = apriori(df_final,
                     min_support=0.05,
                     use_colnames=True)

print(frequentes.head())

   support              itemsets
0     0.10  frozenset({Abóbora})
1     0.07   frozenset({Alface})
2     0.06    frozenset({Arroz})
3     0.09   frozenset({Açúcar})
4     0.15   frozenset({Banana})


In [ ]:
# GERAÇÃO DAS REGRAS
regras = association_rules(frequentes,
                           metric="confidence",
                           min_threshold=0.6)

print(regras.head())

                    antecedents                   consequents  \
0          frozenset({Laranja})           frozenset({Banana})   
1            frozenset({Mamão})           frozenset({Banana})   
2             frozenset({Maçã})           frozenset({Banana})   
3  frozenset({Carne de Frango})     frozenset({Carne Bovina})   
4     frozenset({Carne Bovina})  frozenset({Carne de Frango})   

   antecedent support  consequent support  support  confidence      lift  \
0                0.11                0.15     0.07    0.636364  4.242424   
1                0.11                0.15     0.08    0.727273  4.848485   
2                0.08                0.15     0.06    0.750000  5.000000   
3                0.13                0.13     0.09    0.692308  5.325444   
4                0.13                0.13     0.09    0.692308  5.325444   

   representativity  leverage  conviction  zhangs_metric   jaccard  certainty  \
0               1.0    0.0535    2.337500       0.858748  0.368421   0.

In [ ]:
# FILTRAGEM E ORDENAÇÃO
regras = regras[regras['lift'] > 1]

regras = regras.sort_values(by='lift',
                            ascending=False)

print(regras[['antecedents',
              'consequents',
              'support',
              'confidence',
              'lift']].head())

                    antecedents                   consequents  support  \
4     frozenset({Carne Bovina})  frozenset({Carne de Frango})     0.09   
3  frozenset({Carne de Frango})     frozenset({Carne Bovina})     0.09   
5         frozenset({Linguiça})     frozenset({Carne Bovina})     0.06   
2             frozenset({Maçã})           frozenset({Banana})     0.06   
1            frozenset({Mamão})           frozenset({Banana})     0.08   

   confidence      lift  
4    0.692308  5.325444  
3    0.692308  5.325444  
5    0.666667  5.128205  
2    0.750000  5.000000  
1    0.727273  4.848485  


In [13]:
# RECOMENDAÇÕES INTERPRETADAS
for _, row in regras.head(5).iterrows():

    antecedente = ', '.join(list(row['antecedents']))
    consequente = ', '.join(list(row['consequents']))

    print(f"Se tiver [{antecedente}] → sugerir [{consequente}]")

Se tiver [Carne Bovina] → sugerir [Carne de Frango]
Se tiver [Carne de Frango] → sugerir [Carne Bovina]
Se tiver [Linguiça] → sugerir [Carne Bovina]
Se tiver [Maçã] → sugerir [Banana]
Se tiver [Mamão] → sugerir [Banana]


In [ ]:
# SALVAR RESULTADOS
regras.to_csv('regras_associacao.csv',
              index=False)

print("Arquivo salvo com sucesso.")

Arquivo salvo com sucesso.


# Conclusão

O algoritmo Apriori permitiu identificar padrões relevantes entre alimentos doados nas campanhas do sistema Nação Nutrida. As regras encontradas poderão ser utilizadas futuramente para gerar recomendações automáticas aos usuários da plataforma.